# Fine-Tuning N-ATLaS (8B) on Medical QA / Clinical Dialogue Dataset

This notebook demonstrates fine-tuning **N-ATLaS** (`NCAIR1/N-ATLaS`) using **QLoRA (4-bit Parameter-Efficient Fine-Tuning)** with Hugging Face `transformers`, `peft`, and `trl` (`SFTTrainer`).

### Overview
- **Base Model**: [`NCAIR1/N-ATLaS`](https://huggingface.co/NCAIR1/N-ATLaS) — Nigerian Multilingual Foundation LLM (Llama-3 8B architecture, optimized for English, Hausa, Yoruba, and Igbo).
- **Target Task**: Medical dialogue & clinical consultation assistance (`ruslanmv/ai-medical-chatbot`).
- **Methodology**: 4-bit NormalFloat (NF4) quantization + LoRA target adapters (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).
- **Memory Requirements**: Compatible with Kaggle / Google Colab T4 (16GB) or A10G/L4 GPUs.

## 1. Install Dependencies

Install and upgrade the essential libraries: `transformers`, `datasets`, `accelerate`, `peft`, `trl`, and `bitsandbytes`.

In [ ]:
%%capture
# Install and upgrade dependencies for QLoRA fine-tuning
%pip install -U transformers datasets accelerate peft trl bitsandbytes wandb


## 2. Import Libraries and Modules

Import PyTorch, Hugging Face model and tokenizer utilities, LoRA configurations, and training helpers.

In [ ]:
import os
import torch
import wandb
from datasets import load_dataset
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    logging,
)
from trl import SFTTrainer, setup_chat_format

# Configure logging verbosity
logging.set_verbosity_info()


## 3. Authentication (Hugging Face & Weights & Biases)

Authenticate with Hugging Face Hub (to access base model weights or push adapters) and Weights & Biases (for experiment tracking).

> **Note**: If running on Kaggle, add your secrets in `Add-ons > Secrets`. If running locally or on Colab, replace with environment variables or standard `notebook_login()`.

In [ ]:
from huggingface_hub import login

# Attempt Hugging Face authentication via Kaggle Secrets or environment variable
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
except Exception:
    hf_token = os.environ.get("HUGGINGFACE_TOKEN") or os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Authenticated successfully with Hugging Face Hub.")
else:
    print("No Hugging Face token provided; public checkpoints can still be downloaded.")


In [ ]:
# Weights & Biases experiment tracking setup (optional)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wb_token = user_secrets.get_secret("wandb")
except Exception:
    wb_token = os.environ.get("WANDB_API_KEY")

if wb_token:
    wandb.login(key=wb_token)
    run = wandb.init(
        project="Fine-tune-NATLaS-Medical-Assistant",
        name="natlas-8b-medical-qlora",
        job_type="training",
        anonymous="allow",
    )
else:
    # Disable wandb logging if no API key is configured
    os.environ["WANDB_DISABLED"] = "true"
    print("Weights & Biases logging disabled (no key found).")


## 4. Configuration & Model Selection

We configure the base model to use **N-ATLaS** (`NCAIR1/N-ATLaS`). N-ATLaS is a foundational multilingual LLM built on the Llama-3 8B architecture, developed specifically for Nigerian and regional context while maintaining strong general domain and instruction-following capabilities.

In [ ]:
# Model identifiers and training dataset
base_model = "NCAIR1/N-ATLaS"
dataset_name = "ruslanmv/ai-medical-chatbot"
new_model = "natlas-8b-medical-doctor"

print(f"Base model: {base_model}")
print(f"Dataset:    {dataset_name}")
print(f"Output dir: {new_model}")


## 5. Hardware & Precision Configuration

Detect GPU compute capability to dynamically choose between `bfloat16` with FlashAttention-2 (Ampere / Ada / Hopper architecture, compute capability >= 8.0) and `float16` with SDPA/eager attention for older architectures.

In [ ]:
# Detect GPU capability and select optimal precision and attention implementation
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype = torch.bfloat16
    try:
        import flash_attn  # noqa: F401
        attn_implementation = "flash_attention_2"
    except ImportError:
        attn_implementation = "sdpa"
    print(f"Using bfloat16 precision with {attn_implementation}")
else:
    torch_dtype = torch.float16
    attn_implementation = "sdpa" if hasattr(torch.nn.functional, "scaled_dot_product_attention") else "eager"
    print(f"Using float16 precision with {attn_implementation}")


## 6. Quantization Config & Model Loading

Load N-ATLaS in 4-bit precision using `BitsAndBytesConfig` (NF4, double quantization, 16-bit compute dtype) to allow fine-tuning an 8-billion parameter model on a single commodity GPU with less than 16GB VRAM.

In [ ]:
# QLoRA 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load N-ATLaS base causal language model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation,
    trust_remote_code=True,
)

# Load tokenizer corresponding to N-ATLaS
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


## 7. PEFT / LoRA Adapter Setup

Configure Parameter-Efficient Fine-Tuning (LoRA):
- Rank $r=16$ with scaling factor $\alpha=32$
- Target modules: All linear projection layers in the self-attention and MLP blocks (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`)
- Chat format alignment using `setup_chat_format`

In [ ]:
# LoRA configuration targeting all projection layers
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["up_proj", "down_proj", "gate_proj", "k_proj", "q_proj", "v_proj", "o_proj"],
)

# Harmonize tokenizer chat template and special tokens with model embeddings
model, tokenizer = setup_chat_format(model, tokenizer)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


## 8. Dataset Preparation & Formatting

Load the medical dialogue dataset (`ruslanmv/ai-medical-chatbot`) and format Patient-Doctor exchanges into the standard chat template expected by the tokenizer.

In [ ]:
# Load the medical consultation dataset
dataset = load_dataset(dataset_name, split="all")

# Subsample for fast demonstration / quick training iteration (adjust range as needed)
dataset = dataset.shuffle(seed=65).select(range(1000))

# Map dataset rows into chat template dialogs (user -> Patient, assistant -> Doctor)
def format_chat_template(row):
    dialogue = [
        {"role": "user", "content": row["Patient"]},
        {"role": "assistant", "content": row["Doctor"]},
    ]
    row["text"] = tokenizer.apply_chat_template(dialogue, tokenize=False)
    return row

dataset = dataset.map(format_chat_template, num_proc=2)
print("Sample formatted record preview:")
print(dataset["text"][0][:400] + "...")


## 9. Train / Validation Split

Split the formatted dataset into 90% training and 10% validation sets.

In [ ]:
# Split dataset into 90% training and 10% evaluation
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Training samples:   {len(dataset_split['train'])}")
print(f"Validation samples: {len(dataset_split['test'])}")


## 10. Training Arguments & SFTTrainer Setup

Configure hyperparameters with `TrainingArguments` and initialize `SFTTrainer`. We use `paged_adamw_32bit` optimizer to prevent GPU memory spikes during backpropagation.

In [ ]:
# Configure training hyperparameters
report_to_target = "wandb" if os.environ.get("WANDB_DISABLED") != "true" else "none"

training_arguments = TrainingArguments(
    output_dir=new_model,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    eval_strategy="steps",
    eval_steps=0.2,
    logging_steps=10,
    warmup_steps=10,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=(torch_dtype == torch.float16),
    bf16=(torch_dtype == torch.bfloat16),
    group_by_length=True,
    report_to=report_to_target,
    save_strategy="steps",
    save_steps=0.2,
    save_total_limit=2,
)

# Initialize Supervised Fine-Tuning (SFT) Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_split["train"],
    eval_dataset=dataset_split["test"],
    peft_config=peft_config,
    max_seq_length=512,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
)


## 11. Run Training Loop

Execute the fine-tuning process.

In [ ]:
# Start model training
trainer.train()


## 12. Save Adapters and Artifacts

Save the fine-tuned LoRA adapter weights and tokenizer locally and finalize wandb tracking.

In [ ]:
# Save the fine-tuned LoRA adapters and tokenizer configuration
trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)
model.config.use_cache = True

if wandb.run is not None:
    wandb.finish()

print(f"Model and tokenizer successfully saved to: {new_model}")
# Optional: push adapters to Hugging Face Hub (requires authenticated token with write permissions)
# trainer.model.push_to_hub(new_model, use_temp_dir=False)
# tokenizer.push_to_hub(new_model, use_temp_dir=False)


## 13. Inference & Qualitative Evaluation

Test the fine-tuned medical assistant with patient queries to verify that responses follow medical advice conventions and chat structure.

In [ ]:
# Test Case 1: Dermatological / acne consultation
messages = [{"role": "user", "content": "Hello doctor, I have bad acne. How do I get rid of it?"}]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

response_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("=== Medical Consultation Response 1 ===")
print(response_text.strip())


In [ ]:
# Test Case 2: Systemic fatigue inquiry
messages = [{"role": "user", "content": "Hello doctor, I always feel weak and exhausted, can you help me with that?"}]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

response_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("=== Medical Consultation Response 2 ===")
print(response_text.strip())
